# 03 · Text normalization of device, browser and email strings

The identity table stores user-agent-like strings. The same phone appears as `SM-G930V Build/NRD90M`,
`SAMSUNG SM-G930V` and `SM-G930V`; the same webmail provider as `hotmail.com`, `hotmail.es`, `outlook.com`
and `live.com.mx`. As raw categories each spelling is its own rare level, which splits the signal.

`fraud.text_norm` treats this as a normalization problem: casefold, strip noise tokens (Android build IDs),
map surface variants to a canonical form (brand / family / provider), and keep the informative remainder
(major version, country TLD) as separate features.

In [1]:
import os
import altair as alt
import numpy as np
import pandas as pd

from fraud import charts, db  # charts registers the project's Altair theme
from fraud.config import get_paths
from fraud.io import load_json

PROFILE = os.environ.get("FRAUD_PROFILE", "full")  # "sample" = synthetic data
paths = get_paths(PROFILE)
pd.set_option("display.max_columns", 50, "display.width", 160)
print("profile:", PROFILE)

profile: full


In [2]:
from fraud import text_norm
cov = load_json(paths.reports / "text_norm_coverage.json")
pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "top_unmapped"} for k, v in cov.items()}).T

,raw_levels,normalized_levels,rows_non_null,row_coverage
DeviceInfo -> device_family,1786.0,382.0,118666.0,0.981511
DeviceInfo -> device_brand,1786.0,19.0,118666.0,0.981511
id_30 -> os_family,75.0,6.0,77565.0,0.999678
id_31 -> browser_family,130.0,14.0,140282.0,0.997355
emaildomain -> email_provider,60.0,25.0,633375.0,0.999556


## Examples: many surface forms, one canonical form

In [3]:
conn = db.connect(paths.db, read_only=True)
conn.execute("SET enable_progress_bar = false")
dev = conn.execute("SELECT DeviceInfo AS raw, count(*) AS n, avg(isFraud) AS fraud_rate FROM base "
                   "WHERE DeviceInfo IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 400").df()
dev = dev.join(pd.DataFrame([vars(text_norm.normalize_device(r)) for r in dev.raw]))
dev[dev.device_brand == "samsung"].head(15)

,raw,n,fraud_rate,device_brand,device_family
6,SM-J700M Build/MMB29K,549,0.109290,samsung,samsung j700
7,SM-G610M Build/MMB29K,461,0.080260,samsung,samsung g610
8,SM-G531H Build/LMY48B,410,0.080488,samsung,samsung g531
10,SM-G935F Build/NRD90M,334,0.092814,samsung,samsung g935
11,SM-G955U Build/NRD90M,328,0.100610,samsung,samsung g955
12,SM-G532M Build/MMB29T,316,0.037975,samsung,samsung g532
14,SM-G950U Build/NRD90M,290,0.055172,samsung,samsung g950
15,SM-G930V Build/NRD90M,274,0.000000,samsung,samsung g930
18,SAMSUNG,235,0.021277,samsung,samsung other
19,SM-G950F Build/NRD90M,225,0.151111,samsung,samsung g950


In [4]:
email = conn.execute("SELECT P_emaildomain AS raw, count(*) AS n, avg(isFraud) AS fraud_rate FROM base "
                     "WHERE P_emaildomain IS NOT NULL GROUP BY 1 ORDER BY n DESC").df()
email = email.join(pd.DataFrame([vars(text_norm.normalize_email(r)) for r in email.raw]))
email.groupby("email_provider").agg(raw_spellings=("raw", "count"), n=("n", "sum")).sort_values("n", ascending=False).head(12)

,raw_spellings,n
email_provider,,
google,2,228851
yahoo,9,105969
microsoft,11,59477
anonymous,1,36998
aol,2,28604
att,3,8912
apple,3,8225
comcast,1,7888
verizon,1,2705


## What is still unmapped?

In [5]:
{k: v["top_unmapped"] for k, v in cov.items()}

{'DeviceInfo -> device_family': {'KFFOWI Build/LVY48F': 65,
  'Hisense F23 Build/NRD90M': 61,
  'M4 SS4456 Build/LMY47V': 61,
  'Hisense L675 Build/MRA58K': 60,
  'es-us': 50,
  'Ilium LT510 Build/MRA58K': 46,
  'MDDRJS': 44,
  'Ilium X510 Build/MRA58K': 37,
  'M4 SS4453 Build/MMB29M': 37,
  'KFDOWI Build/LVY48F': 37},
 'DeviceInfo -> device_brand': {'KFFOWI Build/LVY48F': 65,
  'Hisense F23 Build/NRD90M': 61,
  'M4 SS4456 Build/LMY47V': 61,
  'Hisense L675 Build/MRA58K': 60,
  'es-us': 50,
  'Ilium LT510 Build/MRA58K': 46,
  'MDDRJS': 44,
  'Ilium X510 Build/MRA58K': 37,
  'M4 SS4453 Build/MMB29M': 37,
  'KFDOWI Build/LVY48F': 37},
 'id_30 -> os_family': {'other': 15, 'func': 10},
 'id_31 -> browser_family': {'other': 312,
  'Microsoft/Windows': 25,
  'mobile': 11,
  'ZTE/Blade': 9,
  'icedragon': 5,
  'Lanix/Ilium': 3,
  'Cherry': 1,
  'Nokia/Lumia': 1,
  'LG/K-200': 1,
  'BLU/Dash': 1},
 'emaildomain -> email_provider': {'cableone.net': 186, 'ptd.net': 95}}

## Fraud rate by normalized brand / provider

In [6]:
seg = conn.execute("SELECT device_brand, count(*) AS n, avg(isFraud) AS fraud_rate FROM features "
                   "WHERE device_brand IS NOT NULL GROUP BY 1 HAVING count(*) >= 100 ORDER BY fraud_rate DESC").df()
seg

,device_brand,n,fraud_rate
0,nokia,146,0.445205
1,zte,655,0.235115
2,huawei,2610,0.194636
3,lenovo,318,0.147799
4,motorola,3502,0.147059
5,other,2194,0.142662
6,sony,933,0.131833
7,samsung,12131,0.115572
8,alcatel,548,0.113139
9,lg,2563,0.104565


## Does it help the model? (ablation)

In [7]:
res = load_json(paths.reports / "train_results.json")
pol = load_json(paths.reports / "policy_results.json")
cost = {r["model"]: r["total_cost"] for r in pol["by_model"]}
from fraud.report import MODEL_LABELS
rows = []
for k in ("lgbm", "abl_raw_text"):
    v = res["experiments"][k]
    rows.append({"variant": MODEL_LABELS[k], "valid PR-AUC": v["valid"]["pr_auc"], "test PR-AUC": v["test"]["pr_auc"],
                 "n features": v.get("n_features"), "three-tier policy cost": cost.get(MODEL_LABELS[k])})
pd.DataFrame(rows)

,variant,valid PR-AUC,test PR-AUC,n features,three-tier policy cost
0,LightGBM,0.643973,0.518186,476,261045.563
1,Ablation: raw text strings,0.647194,0.522321,463,263925.719


**Result (DRAFT, rewrite after running):** _normalization collapses N raw device strings into M families with
X% row coverage. On the model it moves PR-AUC by ... and policy cost by .... Report this honestly even if the
gain is small: gradient boosting can partly learn raw categories on its own, so the bigger wins are fewer
levels (more robust to unseen spellings in production) and features that are explainable in reason codes._